<a href="https://www.kaggle.com/code/ameythakur20/cross-model-lifecycle-desynchronization" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<br clear="all">

<h1 align="center">Cross-Model Lifecycle Desynchronization</h1>

<p align="center">
  <b>How far does a language model's stale memory override the document in its prompt,<br>
  and does that change with scale or instruction tuning?</b>
</p>

<p align="center">
  <a href="https://github.com/Amey-Thakur/LLM-KNOWLEDGE-LIFECYCLE"><img alt="GitHub" src="https://img.shields.io/badge/GitHub-LLM--KNOWLEDGE--LIFECYCLE-181717?logo=github"></a>
  &nbsp;
  <a href="https://huggingface.co/spaces/ameythakur/llm-knowledge-lifecycle"><img alt="Demo" src="https://img.shields.io/badge/Demo-Hugging_Face_Space-FFD21E?logo=huggingface&logoColor=black"></a>
  &nbsp;
  <a href="https://github.com/Amey-Thakur"><img alt="Authors" src="https://img.shields.io/badge/Authors-Amey_Thakur_%26_Sarvesh_Talele-0969DA"></a>
  &nbsp;
  <a href="https://github.com/Amey-Thakur/LLM-KNOWLEDGE-LIFECYCLE/blob/main/LICENSE"><img alt="License" src="https://img.shields.io/badge/License-CC_BY_4.0-lightgrey"></a>
  &nbsp;
  <a href="https://orcid.org/0000-0001-5644-1575"><img alt="ORCID Amey Thakur" src="https://img.shields.io/badge/ORCID-0000--0001--5644--1575-A6CE39"></a>
  &nbsp;
  <a href="https://orcid.org/0009-0002-0818-461X"><img alt="ORCID Sarvesh Talele" src="https://img.shields.io/badge/ORCID-0009--0002--0818--461X-A6CE39"></a>
</p>

<p align="center">
  <a href="https://github.com/Amey-Thakur/LLM-KNOWLEDGE-LIFECYCLE">Paper</a> &nbsp;&middot;&nbsp;
  <a href="https://huggingface.co/spaces/ameythakur/llm-knowledge-lifecycle">Live demonstration</a> &nbsp;&middot;&nbsp;
  <a href="#method">Method</a> &nbsp;&middot;&nbsp;
  <a href="#results">Results</a> &nbsp;&middot;&nbsp;
  <a href="#findings">Findings</a>
</p>

<hr>

## 1. &nbsp;The question

<br>

A language model carries two memories: the weights it learned during training, and the
document you place in its prompt. When a fact changes after training, those two disagree.

The paper measures that disagreement on GPT-2 for three documented fact changes, using two
numbers per probe.

<br>

| Quantity | Definition | Reading |
| :--- | :--- | :--- |
| **D**<sub>sync</sub> | &minus;ln P(correct answer given the document and the query) | A value of *n* nats means the correct answer holds probability e<sup>&minus;n</sup>. Above 9.2, no realistic decoding recovers it. |
| **I**<sub>ctx</sub> | KL divergence between the with-document and without-document distributions | Near zero means the document is present but inert. |

<br>

Section 9.2 of the paper leaves one question open: **do these numbers improve with model
scale, or with instruction tuning?** This notebook answers it across six open models on the
same three probes.

<br>

> **Environment** &nbsp;&middot;&nbsp; Kaggle, CPU is sufficient, internet on &nbsp;&middot;&nbsp; **Runtime** &nbsp;&middot;&nbsp; under 30 minutes

<br>

---

<a name="method"></a>
## 2. &nbsp;Setup

<br>

Kaggle images ship `torch` and `transformers` already, so nothing is installed here: a run
that resolves its own dependencies can drift between sessions, and this experiment is meant
to give the same numbers every time.

<br>

In [ ]:
import gc
import math

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# CPU by default. This experiment is single forward passes rather than
# generation, so CPU finishes in minutes, spends no GPU quota, and avoids the
# CUDA kernel-image mismatch that some Kaggle images carry. CUDA is adopted
# only when a real operation proves it works.
DEVICE = "cpu"
if torch.cuda.is_available():
    try:
        torch.zeros(4, device="cuda") @ torch.zeros(4, 4, device="cuda")
        DEVICE = "cuda"
    except Exception as error:
        print(f"CUDA present but unusable, staying on CPU: {error}")

print(f"device : {DEVICE}")
print(f"torch  : {torch.__version__}")

<br>

---

## 3. &nbsp;The probes

<br>

Three documented fact changes, byte-identical to the paper and the live demonstration so the
results are directly comparable. Each has a query that ends mid-sentence, a document that
contradicts the model's training-time knowledge, and a single-word verified answer.

<br>

| Probe | Fact change | Regime on GPT-2 (full precision) |
| :--- | :--- | :--- |
| `vioxx` | Rofecoxib withdrawn, September 2004 | Resolution failure &nbsp;&middot;&nbsp; D<sub>sync</sub> 12.05, I<sub>ctx</sub> 0.033 |
| `monarch` | Elizabeth II died, September 2022 | The exposure trap: the document boosts the wrong answer |
| `twitter` | Twitter renamed X, July 2023 | Drift: strong influence, answer still wrong |

<br>

In [ ]:
PROBES = [
    {
        "name": "vioxx",
        "query": "Question: Is Vioxx safe to prescribe? Answer: Vioxx is considered",
        "context": "Context: In September 2004, Merck voluntarily withdrew Vioxx after trials revealed increased cardiovascular risks.",
        "answer": " withdrawn",
    },
    {
        "name": "monarch",
        "query": "Question: Who is the current British monarch? Answer: The current British monarch is",
        "context": "Context: Queen Elizabeth II died in September 2022. Charles III acceded to the throne and is the reigning King of the United Kingdom.",
        "answer": " Charles",
    },
    {
        "name": "twitter",
        "query": "Question: What is the social network Twitter called today? Answer: Twitter is now called",
        "context": "Context: In July 2023, Twitter was rebranded as X under Elon Musk's ownership.",
        "answer": " X",
    },
]

# Two axes in one list: scale inside a single family, where data and tokenizer
# are held constant, and instruction tuning at comparable scale.
MODELS = [
    "gpt2",                                 #  124M  the paper's reference model
    "gpt2-medium",                          #  355M  same family, scale is the only change
    "gpt2-large",                           #  774M
    "Qwen/Qwen2.5-0.5B-Instruct",           #  500M  instruction-tuned, GPT-2-medium scale
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",   #  1.1B  chat-tuned, GPT-2-large scale
    "Qwen/Qwen2.5-1.5B-Instruct",           #  1.5B  largest that stays comfortable on CPU
]

print(f"{len(MODELS)} models x {len(PROBES)} probes = {len(MODELS) * len(PROBES)} measurements")

<br>

---

## 4. &nbsp;The measurement

<br>

One function per idea. The answer's **first token** is what gets measured, exactly as the
paper's estimator specifies, and the tokenization is recorded for every model because each
tokenizer splits a word differently. Logits are cast to float32 before the softmax so that
models stored in half precision are still measured at full precision.

<br>

In [ ]:
def next_token_distribution(model, tokenizer, prompt):
    """Probability of every vocabulary item as the next token."""
    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        logits = model(**inputs).logits[0, -1, :].float()
    return F.softmax(logits, dim=-1)


def run_probe(model, tokenizer, probe):
    """One probe against one model: both conditions, and the two metrics."""
    p_plain = next_token_distribution(model, tokenizer, probe["query"])
    p_ctx = next_token_distribution(model, tokenizer, probe["context"] + "\n" + probe["query"])

    ids = tokenizer.encode(probe["answer"], add_special_tokens=False)
    first_piece = tokenizer.decode([ids[0]])

    # A floor keeps the logarithm finite when a model assigns the answer
    # a probability too small to represent.
    pt_plain = max(p_plain[ids[0]].item(), 1e-12)
    pt_ctx = max(p_ctx[ids[0]].item(), 1e-12)

    return {
        "probe": probe["name"],
        "answer_first_piece": repr(first_piece),
        "answer_pieces": len(ids),
        "P_answer_plain": pt_plain,
        "P_answer_ctx": pt_ctx,
        "D_sync_nats": -math.log(pt_ctx),
        "I_ctx_nats": F.kl_div(p_plain.log(), p_ctx, reduction="sum").item(),
        "top_token_plain": repr(tokenizer.decode([int(torch.argmax(p_plain))])),
        "top_token_ctx": repr(tokenizer.decode([int(torch.argmax(p_ctx))])),
        "correct_top1": tokenizer.decode([int(torch.argmax(p_ctx))]) == first_piece,
    }

<br>

---

## 5. &nbsp;Running the sweep

<br>

Models are loaded one at a time and released before the next. Holding two of them at once
is what exhausts memory on a free instance, and nothing here needs them simultaneously.

<br>

In [ ]:
rows = []

for name in MODELS:
    print(f"\n{name}")
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name,
        torch_dtype="auto" if DEVICE == "cuda" else torch.float32,
        device_map=DEVICE,
    )
    model.eval()
    size_m = round(sum(p.numel() for p in model.parameters()) / 1e6)

    for probe in PROBES:
        row = {"model": name, "params_M": size_m}
        row.update(run_probe(model, tokenizer, probe))
        rows.append(row)
        print(f"   {row['probe']:<9} D_sync {row['D_sync_nats']:6.2f}   "
              f"I_ctx {row['I_ctx_nats']:6.3f}   correct_top1 {row['correct_top1']}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results.to_csv("cross_model_dsync.csv", index=False)
print(f"\n{len(results)} measurements written to cross_model_dsync.csv")

<br>

---

## 6. &nbsp;Reproduction check

<br>

The `gpt2` and `vioxx` row must reproduce the paper's published measurement before anything
downstream is trusted: **D**<sub>sync</sub> = 12.0464 and **I**<sub>ctx</sub> = 0.0333. The
tolerance allows for library-version noise in the last digits while still catching a real
deviation.

<br>

> **If this cell fails, stop and report the numbers.** A sweep that does not reproduce its
> own reference point cannot support a conclusion.

<br>

In [ ]:
check = results[(results.model == "gpt2") & (results.probe == "vioxx")].iloc[0]

print(f"D_sync  measured {check.D_sync_nats:8.4f}   paper 12.0464")
print(f"I_ctx   measured {check.I_ctx_nats:8.4f}   paper  0.0333")

assert abs(check.D_sync_nats - 12.0464) < 0.05, "does not reproduce the paper's measurement"
print("\nreproduction check passed")

<br>

---

<a name="results"></a>
## 7. &nbsp;Results

<br>

One table per metric, models in ascending size, probes as columns. These are the tables that
go back into the paper, so they are built from the measurements rather than typed by hand.

<br>

In [ ]:
for metric, title in [
    ("D_sync_nats", "D_sync (nats) - lower is better"),
    ("I_ctx_nats", "I_ctx (nats) - how far the document moved the model"),
    ("correct_top1", "Did the model actually answer correctly?"),
]:
    print(f"\n{title}")
    print("-" * len(title))
    print(results.pivot(index="model", columns="probe", values=metric)
                 .reindex(MODELS).round(3).to_string())

<br>

---

<a name="findings"></a>
## 8. &nbsp;Findings

<br>

*Written after the run, from the tables above. One short paragraph per question, and every
sentence carries the number that supports it. Nothing that failed is deleted: a model that
behaves strangely is a finding, not a blemish.*

<br>

**1. &nbsp;Scale within one family.** &nbsp; Does D<sub>sync</sub> fall from `gpt2` to
`gpt2-large` on each probe, and by how much?

<br>

**2. &nbsp;Instruction tuning.** &nbsp; Do the tuned models resolve the conflict, meaning
`correct_top1` is true, where base models of similar size do not?

<br>

**3. &nbsp;The exposure trap.** &nbsp; On `monarch`, does any model repeat the paper's
pattern, where the corrective document raises the probability of the *wrong* answer? Compare
`P_answer_plain` against `P_answer_ctx`, and watch `top_token_ctx`.

<br>

<hr>

<p align="center">
  <b>Amey Thakur</b> &nbsp;&middot;&nbsp; <b>Sarvesh Talele</b> &nbsp;&middot;&nbsp; Independent Research
</p>

<p align="center">
  Attach <code>cross_model_dsync.csv</code> alongside the written findings.
</p>